# 🏠 Rental Agreement Risk Analyzer (Pune)
**Subject:** Natural Language Processing Lab  |  **Name:** ________  |  **Roll No.:** ________  |  **Date:** ________

**Aim:** To build an NLP system that reads a rental (Leave & License) agreement, splits it into clauses, extracts key facts (rent, deposit, notice period, area), classifies each clause and rates its **risk (Low / Medium / High)** using Pune market norms.

**How to use this notebook:** Run the cells **top to bottom** (Runtime ▸ Run all, or Shift+Enter on each cell).
Every cell marked **📸** produces output you should screenshot for your report.

## 0. Setup
Installs the libraries and downloads NLTK data and the spaCy English model (~1 minute).

In [ ]:
!pip install -q nltk pandas matplotlib spacy ipywidgets
!python -m spacy download en_core_web_sm -q

import nltk
for r in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(r, quiet=True)
print("Setup complete ✅")

## STEP 1 — Problem Statement, Pune Rules & Dataset

**Problem:** Tenants often sign rental agreements without understanding risky clauses (unfair deposit deductions, eviction without notice, huge rent hikes). This project flags such clauses automatically.

**Key idea (hybrid design):**
- *Numeric clauses* (rent, deposit, % increase, late fee) → judged by **rules calibrated to Pune market rates**.
- *Wording-based clauses* (eviction, lock-in, maintenance) → judged by an **ML classifier** (Step 4).

The cell below saves our Pune rules into one file (`pune_rules.py`) so every step uses the same numbers.

In [ ]:
%%writefile pune_rules.py
"""
pune_rules.py  -  Pune rental-market rules (single source of truth)
Used by: dataset generator (labels), Step 3 (parsing) and Step 5 (risk scoring).
All figures are from the student's own market research (Pune, 2026) unless
marked ASSUMPTION.
"""

# Monthly rent ranges (Rs.) -> {area: {bhk: (min, max)}}
RENT_RANGES = {
    "kothrud":            {1: (18000, 26000), 2: (27000, 38000), 3: (38000, 55000)},
    "hinjewadi phase 1":  {1: (13000, 20000), 2: (22000, 32000), 3: (30000, 43000)},
    "hinjewadi phase 2/3":{1: (13000, 21000), 2: (23000, 35000), 3: (32000, 48000)},
    "wakad":              {1: (17000, 25000), 2: (26000, 36000), 3: (38000, 55000)},
    "viman nagar":        {1: (22000, 32000), 2: (32000, 45000), 3: (45000, 65000)},
    "baner/balewadi":     {1: (20000, 30000), 2: (32000, 45000), 3: (45000, 70000)},
    "kharadi":            {1: (18000, 28000), 2: (28000, 40000), 3: (40000, 60000)},
}
RENT_RANGES["sus gaon"] = RENT_RANGES["hinjewadi phase 2/3"]   # same as Hinjewadi Ph 2/3 (given)

# Words found in text -> key in RENT_RANGES
AREA_ALIASES = {
    "kothrud": "kothrud", "wakad": "wakad", "kharadi": "kharadi",
    "viman nagar": "viman nagar", "baner": "baner/balewadi", "balewadi": "baner/balewadi",
    "sus gaon": "sus gaon", "sus": "sus gaon", "susgaon": "sus gaon",
    "hinjewadi phase 1": "hinjewadi phase 1", "hinjewadi phase 2": "hinjewadi phase 2/3",
    "hinjewadi phase 3": "hinjewadi phase 2/3", "hinjewadi": "hinjewadi phase 2/3",  # plain Hinjewadi -> Ph 2/3
}

# Society maintenance (Rs./month) - for reference in the report
SOCIETY_MAINTENANCE = {"basic": (1500, 3000), "standard": (3000, 6000), "premium": (6000, 12000)}

def rent_risk(area_key, bhk, rent):
    """Extra is counted ABOVE THE START (lowest rent) of the area/BHK range (confirmed by the student):
       up to Rs.4,000 above the start -> Low ; up to Rs.7,000 -> Medium ; more than Rs.7,000 -> High.
       Rent at or below the start is Low. Facilities / furnishing are ignored."""
    lo, hi = RENT_RANGES[area_key][bhk]
    over = rent - lo
    if over <= 4000:  return "Low"
    if over <= 7000:  return "Medium"
    return "High"

def deposit_risk(deposit, rent):
    """1-2x rent Low, up to 3x Medium, above 3x High."""
    m = deposit / rent
    if m <= 2: return "Low"
    if m <= 3: return "Medium"
    return "High"

def increase_risk(pct):
    """<=10% Low, 10-15% Medium, >15% High (>=20% flagged excessive)."""
    if pct <= 10: return "Low"
    if pct <= 15: return "Medium"
    return "High"

def refund_risk(days):
    """Refund within 15-30 days Low; 31-60 Medium; later High.   [60-day cut-off = ASSUMPTION]"""
    if days <= 30: return "Low"
    if days <= 60: return "Medium"
    return "High"

def latefee_risk(grace_days, fee_per_day):
    """Modest fee after a 5-7 day grace period is normal.
       ASSUMPTION: 'modest' = up to Rs.200/day; Medium up to Rs.500/day."""
    if grace_days >= 5 and fee_per_day <= 200: return "Low"
    if grace_days >= 3 and fee_per_day <= 500: return "Medium"
    return "High"

def lockin_risk(months, one_sided):
    """One-sided lock-in -> High. Mutual: <=6 months Low, 7-11 Medium, >11 High.  [ASSUMPTION - see list below]"""
    if one_sided: return "High"
    if months <= 6: return "Low"
    if months <= 11: return "Medium"
    return "High"

# Who normally pays what (from the student's notes) - used in Step 5 keyword rules
LANDLORD_PAYS = ["society maintenance", "sinking fund", "corpus fund", "structural repair"]
TENANT_PAYS   = ["electricity", "gas", "internet", "metered water"]
NEGOTIABLE    = ["clubhouse", "move-in", "move-out", "appliance servicing"]

ASSUMPTIONS = [
    "Rent rule (CONFIRMED by student, Reading B): extra is measured above the START (lowest rent) of the area/BHK range.",
    "Late fee: 'modest' taken as <= Rs.200/day (Low) and <= Rs.500/day (Medium).",
    "Deposit refund: > 60 days treated as High.",
    "Lock-in: <= 6 months mutual = Low; 7-11 months = Medium; one-sided or forfeiture = High.",
]

Now the dataset generator. Numeric clauses are **generated from templates and labelled by the rules**; qualitative clauses are **hand-labelled**.

In [ ]:
%%writefile make_dataset.py
# STEP 1 (v2): Pune-calibrated labelled dataset.
# Numeric clauses are GENERATED from templates and labelled by pune_rules.py
# (so labels always follow the rules). Qualitative clauses are hand-labelled.
import csv, random
from pune_rules import *

random.seed(42)
rows = []

def indian(n):                      # 126000 -> 1,26,000
    s = str(int(n))
    if len(s) <= 3: return s
    head, tail = s[:-3], s[-3:]
    parts = []
    while len(head) > 2: parts.insert(0, head[-2:]); head = head[:-2]
    if head: parts.insert(0, head)
    return ",".join(parts + [tail])

def mid(area, bhk):
    lo, hi = RENT_RANGES[area][bhk]; return int(round((lo + hi) / 2 / 500) * 500)

AREAS = [("Kothrud","kothrud"),("Hinjewadi Phase 1","hinjewadi phase 1"),("Hinjewadi Phase 2","hinjewadi phase 2/3"),
         ("Sus Gaon","sus gaon"),("Wakad","wakad"),("Viman Nagar","viman nagar"),("Baner","baner/balewadi"),
         ("Balewadi","baner/balewadi"),("Kharadi","kharadi")]

# ---- 1. RENT (generated, 14) ----
offsets = [-1000, 0, 1500, 2500, 3500, 4000, 4500, 5500, 6500, 7000, 7500, 9000, 12000, 15000]   # measured from the START of the range
for i, off in enumerate(offsets):
    disp, key = AREAS[i % len(AREAS)]; bhk = [1, 2, 2, 3][i % 4]
    lo, hi = RENT_RANGES[key][bhk]
    rent = int(round((lo + off) / 500) * 500)
    text = random.choice([
        f"The monthly rent for the {bhk} BHK flat at {disp} shall be Rs. {indian(rent)}, payable on or before the fifth of every month.",
        f"The Tenant agrees to pay a monthly rent of Rs. {indian(rent)} for the {bhk} BHK flat situated in {disp}."])
    rows.append((text, "rent", rent_risk(key, bhk, rent), "rule"))

# ---- 2. DEPOSIT amount (generated, 12) ----
for i, mult in enumerate([1, 1.5, 2, 2, 2.5, 3, 3, 3.5, 4, 5, 6, 10]):
    disp, key = AREAS[(i + 2) % len(AREAS)]; bhk = [2, 1, 3][i % 3]
    rent = int(round((RENT_RANGES[key][bhk][0] + 1000) / 500) * 500); dep = int(rent * mult)   # rent kept low-risk so the label reflects the deposit only
    text = f"The monthly rent for the {bhk} BHK flat at {disp} is Rs. {indian(rent)} and the Tenant shall pay a refundable security deposit of Rs. {indian(dep)}."
    rows.append((text, "deposit", deposit_risk(dep, rent), "rule"))

# ---- 3. DEPOSIT refund time (generated, 5) ----
for d in [15, 30, 45, 60, 90]:
    w = {15:"fifteen",30:"thirty",45:"forty five",60:"sixty",90:"ninety"}[d]
    rows.append((f"The security deposit shall be refunded within {w} days of handover after deducting documented damages.", "deposit", refund_risk(d), "rule"))

# ---- 4. RENT INCREASE (generated, 10) ----
for p in [5, 5, 7, 8, 10, 12, 15, 18, 20, 25]:
    rows.append((f"The rent shall be increased by {p} percent on renewal of the agreement after every eleven months.", "rent_increase", increase_risk(p), "rule"))

# ---- 5. LATE FEE (generated, 8) ----
for grace, fee in [(7,100),(5,200),(7,150),(5,400),(3,300),(2,200),(0,500),(3,1000)]:
    rows.append((f"A late fee of Rs. {fee} per day shall be charged if the rent is not paid within {grace} days of the due date." if grace
                 else f"A late fee of Rs. {fee} per day shall be charged from the day after the due date, with no grace period.",
                 "penalty", latefee_risk(grace, fee), "rule"))

# ---- 6. HAND-LABELLED qualitative clauses ----
H = []
def add(t, typ, r): H.append((t, typ, r, "hand"))
# deposit
add("The Landlord may deduct any amount from the deposit at his sole discretion without providing proof.", "deposit", "High")
add("The security deposit shall not be refunded if the Tenant leaves before completing the full term, whatever the reason.", "deposit", "High")
add("The deposit shall be refunded after deducting only documented damages, and the bills shall be shown to the Tenant.", "deposit", "Low")
add("The Landlord may retain the entire deposit as a cleaning and repainting charge irrespective of the actual condition.", "deposit", "High")
# lock-in
add("The agreement is a registered Leave and License Agreement for eleven months with a mutual lock-in of three months for both parties.", "lockin", "Low")
add("Either party may terminate during the lock-in period of six months by giving one month's written notice and paying one month's rent.", "lockin", "Low")
add("The Tenant shall not vacate during the lock-in period of eleven months, failing which the entire remaining rent becomes payable immediately.", "lockin", "High")
add("There is a lock-in period of twelve months applicable only to the Tenant, while the Landlord may terminate at any time.", "lockin", "High")
add("The lock-in period is eight months, and early exit requires payment of two months' rent as compensation.", "lockin", "Medium")
add("If the Tenant leaves within the lock-in period, the security deposit shall be forfeited and the balance rent shall be recovered.", "lockin", "High")
add("A lock-in period of six months applies to both parties, after which one month's notice is sufficient.", "lockin", "Low")
add("Early termination during the lock-in period is possible only with the written consent of the Landlord, which may be withheld.", "lockin", "Medium")
add("The tenancy is for eleven months and may be renewed by mutual written consent.", "lockin", "Low")
add("The lock-in period is nine months for the Tenant, and the Landlord has no lock-in.", "lockin", "High")
# eviction
add("The Landlord may terminate this agreement only on breach of terms, after giving a written notice of thirty days to remedy it.", "eviction", "Low")
add("Eviction shall follow due process of law, and the Tenant will be given a minimum of sixty days to vacate.", "eviction", "Low")
add("The Landlord reserves the right to evict the Tenant at any time without notice and without assigning any reason.", "eviction", "High")
add("The Tenant must vacate within seven days of receiving a notice from the Landlord, failing which belongings may be removed.", "eviction", "High")
add("The Landlord can terminate the tenancy by giving fifteen days' notice if the property is required for personal use.", "eviction", "Medium")
add("In case of non-payment of rent for two consecutive months, the Landlord may terminate the agreement after a seven day notice.", "eviction", "Medium")
add("The Landlord may enter and lock the premises immediately if the Tenant delays rent by even one day.", "eviction", "High")
add("Either party must give a written notice of one month before ending the tenancy.", "eviction", "Low")
add("On termination for personal use, the Tenant shall get thirty days' notice and the deposit shall be refunded on the same day.", "eviction", "Low")
# maintenance
add("The Landlord shall pay the society maintenance, sinking fund and corpus fund, and shall bear the cost of structural repairs.", "maintenance", "Low")
add("The Tenant shall pay the society maintenance as per the latest society bill, which the Landlord has shown to the Tenant.", "maintenance", "Medium")
add("Monthly maintenance shall be paid extra by the Tenant, as informed by the Landlord from time to time.", "maintenance", "High")
add("The Tenant shall bear the sinking fund, corpus fund and any structural repairs of the flat.", "maintenance", "High")
add("Any damage to the property, including normal wear and tear, shall be repaired at the Tenant's expense.", "maintenance", "High")
add("Clubhouse, move-in and move-out charges shall be shared equally between the Landlord and the Tenant.", "maintenance", "Medium")
add("Minor repairs caused by the Tenant shall be done at his cost, while major repairs not caused by the Tenant shall be done by the Landlord within seven days.", "maintenance", "Low")
add("The Landlord is not responsible for any repairs during the tenancy and the Tenant must not complain about defects.", "maintenance", "High")
add("Appliance servicing shall be done by the Tenant at his own cost.", "maintenance", "Medium")
add("The Tenant shall repaint the entire flat before vacating at his own cost.", "maintenance", "Medium")
# utilities
add("Electricity and metered water shall be paid by the Tenant as per the actual meter readings.", "utilities", "Low")
add("The Tenant shall pay electricity at the tariff charged by MSEDCL without any markup.", "utilities", "Low")
add("Electricity shall be charged at Rs. 15 per unit as decided by the Landlord, irrespective of the actual tariff.", "utilities", "High")
add("Gas, internet and metered water charges shall be paid by the Tenant as per actual bills.", "utilities", "Low")
add("The Tenant shall pay a fixed monthly charge of Rs. 800 for water irrespective of consumption.", "utilities", "Medium")
add("All property tax and building taxes shall be paid by the Tenant in addition to the monthly rent.", "utilities", "High")
add("The Landlord may cut off electricity and water supply if rent is delayed by any number of days.", "utilities", "High")
add("Common area electricity and lift charges shall be paid through the society maintenance borne by the Landlord.", "utilities", "Low")
add("Internet and cable charges shall be paid by the Tenant, and the Landlord shall permit the installation.", "utilities", "Low")
# penalty (qualitative)
add("The Tenant shall pay a penalty of Rs. 5,000 per day for any breach of any clause of this agreement.", "penalty", "High")
add("A penalty equal to three months' rent will be imposed if guests stay overnight without permission.", "penalty", "High")
add("The Landlord may impose fines at his discretion for noise, parking or any other issue as he deems fit.", "penalty", "High")
add("No late fee shall be charged if the rent is paid within a grace period of seven days after the due date.", "penalty", "Low")
# general / boilerplate clauses (written in my own words, not copied from any template)
add("The Licensee shall use the flat only for residential purposes and not for any business or commercial activity.", "general", "Low")
add("The Licensee shall not sublet, assign or part with possession of the flat or any part of it to any other person.", "general", "Low")
add("No structural change, drilling or alteration shall be made in the flat without the written consent of the Licensor.", "general", "Low")
add("The Licensee shall keep the premises clean and shall not cause nuisance or annoyance to the neighbours.", "general", "Low")
add("The Licensor or his representative may inspect the flat at reasonable times after giving prior notice to the Licensee.", "general", "Low")
add("The Licensee shall comply with the rules and bye-laws of the housing society.", "general", "Low")
add("The Licensee shall not store any hazardous, illegal or inflammable goods in the flat.", "general", "Low")
add("The Licensee shall hand over the flat in the same condition as received, excluding normal wear and tear.", "general", "Low")
add("The Licensee shall not keep pets without the prior permission of the Licensor and the society.", "general", "Low")
add("The Licensor confirms that he is the lawful owner of the flat and has the right to give it on leave and license.", "general", "Low")
add("This agreement shall be governed by the laws of India and the courts at Pune shall have jurisdiction.", "general", "Low")
add("The Licensor shall hand over vacant possession of the flat to the Licensee on the date of commencement.", "general", "Low")
add("The Licensee shall not allow more than the agreed number of persons to stay in the flat.", "general", "Low")
add("Overnight visitors are allowed only with the prior written permission of the Licensor, who may refuse without giving reasons.", "general", "Medium")
add("The Licensor may enter the flat at any time without informing the Licensee.", "general", "High")
add("The Licensee shall be fully liable for every loss or damage to the flat, whether or not it was caused by the Licensee.", "general", "High")
rows += H

random.shuffle(rows)
with open("clauses.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["clause_text", "clause_type", "risk_level", "label_source"]); w.writerows(rows)

import collections
print("Saved", len(rows), "clauses")
print("Types:", dict(collections.Counter(r[1] for r in rows)))
print("Risk :", dict(collections.Counter(r[2] for r in rows)))

In [ ]:
!python make_dataset.py

### 📸 Dataset overview

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv("clauses.csv")
display(df.sample(8, random_state=1))
print("Total clauses:", len(df))
display(pd.crosstab(df.clause_type, df.risk_level))

ax = pd.crosstab(df.clause_type, df.risk_level)[["Low","Medium","High"]].plot(
        kind="bar", stacked=True, color=["#4caf50","#ffc107","#e53935"], figsize=(8,4))
ax.set_title("Dataset: clause types vs risk level"); ax.set_ylabel("Number of clauses")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

### 📸 Pune rule engine — quick tests (good for your viva)

In [ ]:
from pune_rules import *
print("Kothrud 2BHK (range starts 27,000): rent 29,000 ->", rent_risk("kothrud", 2, 29000))   # +2,000 above start
print("Kothrud 2BHK, rent 33,000 ->", rent_risk("kothrud", 2, 33000))   # +6,000 above start
print("Kothrud 2BHK, rent 36,000 ->", rent_risk("kothrud", 2, 36000))   # +9,000 above start
print("Deposit 72,000 on rent 36,000 (2x)   ->", deposit_risk(72000, 36000))
print("Deposit 1,26,000 on rent 36,000 (3.5x)->", deposit_risk(126000, 36000))
print("Rent increase 12% ->", increase_risk(12), "| 20% ->", increase_risk(20))
print("\nAssumptions made:"); [print(" -", a) for a in ASSUMPTIONS];

## STEP 2 — Text Preprocessing with NLTK  *(Practical 1)*

Pipeline: **lowercase → regex cleaning → tokenization → stop-word removal → lemmatization**

Design choices to remember for the viva:
1. **Placeholders, not deletion** — `Rs. 50,000` → `moneyamt`; `5 percent` → `percentval`. The classifier learns "money is involved" without caring about the exact value.
2. **Protected stop-words** — *no, not, nor, never, only, before, after, against* are **kept**. "shall **not** be refunded" ≠ "shall be refunded".
3. **Lemmatization over stemming** — gives real words (`security`) instead of `secur`.

In [ ]:
import re
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# ---------- Setup ----------
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True); nltk.download("wordnet", quiet=True)

# KEY IDEA: normal stop-word lists delete words like "no", "not", "without".
# In a contract, "may evict WITHOUT notice" is very different from "may evict with notice".
# So we protect these words from removal.
KEEP = {"no", "not", "nor", "never", "without", "before", "after", "only", "against"}
STOP = set(stopwords.words("english")) - KEEP

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def normalise(text: str) -> str:
    """Regex-based cleaning. We replace values with placeholder tokens instead of
    deleting them, because 'a money amount is deducted' still carries meaning."""
    t = text.lower()                                              # 1. lowercase
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)                  # 2. remove URLs
    t = re.sub(r"(rs\.?|inr|₹)\s*[\d,]+(\.\d+)?", " moneyamt ", t)  # 3. Rs. 50,000 -> moneyamt
    t = re.sub(r"\d+(\.\d+)?\s*(percent|%)", " percentval ", t)   # 4. 5 percent -> percentval
    t = re.sub(r"\b\d+\b", " numval ", t)                         # 5. other numbers -> numval
    t = re.sub(r"[^a-z\s]", " ", t)                               # 6. remove punctuation/symbols
    t = re.sub(r"\s+", " ", t).strip()                            # 7. collapse spaces
    return t


def preprocess(text: str, use_lemma: bool = True) -> str:
    t = normalise(text)
    tokens = word_tokenize(t)                                     # tokenization
    tokens = [w for w in tokens if w not in STOP and len(w) > 1]  # stop-word removal
    if use_lemma:
        tokens = [lemmatizer.lemmatize(w, pos="v") for w in tokens]  # lemmatize as verb ("charged"->"charge")
        tokens = [lemmatizer.lemmatize(w, pos="n") for w in tokens]  # then as noun ("deductions"->"deduction")
    else:
        tokens = [stemmer.stem(w) for w in tokens]
    return " ".join(tokens)

### 📸 Stage-by-stage output on one clause

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize
df = pd.read_csv("clauses.csv")
sample = df.loc[df.clause_text.str.contains(" not ")].iloc[0].clause_text

print("ORIGINAL        :", sample)
norm = normalise(sample);                       print("\n1. AFTER REGEX  :", norm)
toks = word_tokenize(norm);                     print("2. WORD TOKENS  :", toks)
no_stop = [w for w in toks if w not in STOP and len(w) > 1]
print("3. STOP REMOVED :", no_stop)
print("   protected (kept) words:", [w for w in toks if w in stopwords.words('english') and w not in STOP])
print("4a. STEMMING    :", [stemmer.stem(w) for w in no_stop])
print("4b. LEMMATIZED  :", preprocess(sample).split())

### 📸 Stemming vs Lemmatization

In [ ]:
words = ["deductions", "charged", "terminated", "evicting", "premises", "responsibility", "refunded"]
pd.DataFrame({
    "word": words,
    "Porter stem": [stemmer.stem(w) for w in words],
    "lemma (verb)": [lemmatizer.lemmatize(w, "v") for w in words],
    "lemma (noun)": [lemmatizer.lemmatize(w, "n") for w in words],
})

### 📸 Apply to the whole dataset

In [ ]:
df["clean_text"] = df["clause_text"].apply(preprocess)
df["tokens_before"] = df["clause_text"].apply(lambda s: len(word_tokenize(s)))
df["tokens_after"]  = df["clean_text"].apply(lambda s: len(s.split()))
df.to_csv("clauses_clean.csv", index=False)

display(df[["clause_text", "clean_text"]].head(4).style.set_properties(**{"text-align": "left"}))
print(f"Average tokens per clause: {df.tokens_before.mean():.1f} → {df.tokens_after.mean():.1f} "
      f"({100*(1-df.tokens_after.mean()/df.tokens_before.mean()):.0f}% reduction)")

plt.figure(figsize=(5,3.5))
plt.bar(["Before","After"], [df.tokens_before.mean(), df.tokens_after.mean()], color=["#90a4ae","#1e88e5"])
plt.title("Avg tokens per clause"); plt.ylabel("tokens"); plt.show()

## STEP 3 — Clause Splitting & spaCy Analysis  *(Practical 2)*

1. **Clause splitter** – NLTK's sentence tokenizer wrongly cuts at `Rs.`; we split on clause numbers and protect abbreviations.
2. **spaCy** – tokens, POS tags, lemmas, dependencies, noun chunks, named entities.
3. **Hybrid NER** – pretrained spaCy NER + our own `EntityRuler` rules for domain entities: `MONEY_AMT, PERCENT_VAL, DURATION, AREA, BHK, PARTY`.
4. **Value parsers** – `"Rs. 1,26,000"` → `126000`, `"forty five days"` → `45`.

First, a sample agreement to analyse (you can replace this with a real one later):

In [ ]:
with open("sample_agreement.txt", "w", encoding="utf-8") as f:
    f.write('''LEAVE AND LICENSE AGREEMENT

This agreement is made at Pune on 1 November 2026 between Mr. Rajesh Kulkarni (the Landlord) and Ms. Ananya Joshi (the Tenant).

1. The monthly rent for the 2 BHK flat at Kothrud shall be Rs. 29,000, payable on or before the fifth of every month.
2. The Tenant shall pay a refundable security deposit of Rs. 1,01,500 to the Landlord on signing this agreement.
3. The security deposit shall be refunded within forty five days of handover after deducting documented damages.
4. The rent shall be increased by 12 percent on renewal of the agreement after every eleven months.
5. There is a lock-in period of twelve months applicable only to the Tenant, while the Landlord may terminate at any time.
6. The Landlord reserves the right to evict the Tenant at any time without notice and without assigning any reason.
7. Monthly maintenance shall be paid extra by the Tenant, as informed by the Landlord from time to time.
8. Electricity and metered water shall be paid by the Tenant as per the actual meter readings.
9. A late fee of Rs. 500 per day shall be charged from the day after the due date, with no grace period.
''')
print(open("sample_agreement.txt", encoding="utf-8").read())

In [ ]:
import re
import spacy
from nltk.tokenize import sent_tokenize
from pune_rules import AREA_ALIASES

In [ ]:
# ---- Clause splitter ----
ABBR = ["Rs.", "Mr.", "Ms.", "Mrs.", "Dr.", "No.", "approx.", "i.e.", "e.g."]

def split_clauses(text):
    """Split an agreement into clauses. Numbered clauses ('1.', '2)') are split by regex;
       otherwise fall back to sentence splitting with abbreviations protected."""
    parts = re.split(r"(?m)^\s*\d+[.)]\s+", text)       # split at line-start numbering
    header, clauses = parts[0].strip(), [p.strip() for p in parts[1:] if p.strip()]
    if clauses:
        return header, clauses
    t = text
    for a in ABBR:                                       # protect the dots
        t = t.replace(a, a.replace(".", "<DOT>"))
    sents = [s.replace("<DOT>", ".").strip() for s in sent_tokenize(t)]
    return "", sents

In [ ]:
# ---- spaCy pipelines: pretrained vs hybrid ----
nlp_base = spacy.load("en_core_web_sm")                  # pretrained only
nlp = spacy.load("en_core_web_sm")                       # pretrained + our rules
ruler = nlp.add_pipe("entity_ruler", before="ner", config={"overwrite_ents": True})

UNITS = ["day", "days", "month", "months", "year", "years"]
patterns = [
    {"label": "MONEY_AMT",   "pattern": [{"LOWER": {"REGEX": r"^(rs\.?|inr|₹)$"}}, {"ORTH": ".", "OP": "?"}, {"LIKE_NUM": True}]},
    {"label": "PERCENT_VAL", "pattern": [{"LIKE_NUM": True}, {"LOWER": {"IN": ["percent", "%"]}}]},
    {"label": "DURATION",    "pattern": [{"LIKE_NUM": True, "LOWER": {"NOT_IN": ["first", "second", "third", "fourth", "fifth", "next"]}, "OP": "+"}, {"LOWER": {"IN": UNITS}}]},
    {"label": "BHK",         "pattern": [{"LIKE_NUM": True}, {"LOWER": "bhk"}]},
    {"label": "BHK",         "pattern": [{"LOWER": {"REGEX": r"^\d+bhk$"}}]},
    {"label": "PARTY",       "pattern": [{"LOWER": {"IN": ["landlord", "tenant", "licensor", "licensee"]}}]},
]
# Pune localities (multi-word names written as token patterns)
for area in ["kothrud", "wakad", "kharadi", "baner", "balewadi", "hinjewadi", "viman nagar", "sus gaon"]:
    patterns.append({"label": "AREA", "pattern": [{"LOWER": w} for w in area.split()]})
patterns.append({"label": "AREA", "pattern": [{"LOWER": "hinjewadi"}, {"LOWER": "phase"}, {"LIKE_NUM": True}]})
ruler.add_patterns(patterns)

In [ ]:
# ---- Value parsers & fact extraction ----
WORDS = {"zero":0,"one":1,"two":2,"three":3,"four":4,"five":5,"six":6,"seven":7,"eight":8,"nine":9,"ten":10,
         "eleven":11,"twelve":12,"thirteen":13,"fourteen":14,"fifteen":15,"sixteen":16,"seventeen":17,
         "eighteen":18,"nineteen":19,"twenty":20,"thirty":30,"forty":40,"fifty":50,"sixty":60,"seventy":70,
         "eighty":80,"ninety":90}

def words_to_number(s):
    s = s.lower().replace("-", " ").strip()
    if re.fullmatch(r"[\d,\.]+", s): return float(s.replace(",", ""))
    toks = [WORDS[w] for w in s.split() if w in WORDS]
    return float(sum(toks)) if toks else None

def first_number(s):                      # "12 percent" / "10%" / "ten percent" -> number
    m = re.search(r"\d+(\.\d+)?", s)
    if m: return float(m.group())
    return words_to_number(re.sub(r"[^a-z\s-]", "", s.lower()).split()[0])

def money_value(s):                       # "Rs. 1,26,000" -> 126000
    m = re.search(r"[\d,]+(\.\d+)?", s); return int(float(m.group().replace(",", ""))) if m else None

def duration_days(s):                     # "forty five days" -> 45 ; "eleven months" -> 330
    m = re.match(r"(.+?)\s*(days?|months?|years?)$", s.strip().lower())
    n = words_to_number(m.group(1)); u = m.group(2)
    if n is None: return 0                # unparseable (e.g. ordinal words) -> ignore safely
    return int(n * (1 if u.startswith("day") else 30 if u.startswith("month") else 365))

def extract_facts(clause):
    """Return a dict of structured facts found in one clause."""
    doc = nlp(clause); f = {"money": [], "percent": [], "durations_days": [], "area": None, "bhk": None, "parties": set()}
    for e in doc.ents:
        if e.label_ == "MONEY_AMT":   f["money"].append(money_value(e.text))
        elif e.label_ == "PERCENT_VAL": f["percent"].append(first_number(e.text))
        elif e.label_ == "DURATION":  f["durations_days"].append(duration_days(e.text))
        elif e.label_ == "BHK":       f["bhk"] = int(first_number(e.text))
        elif e.label_ == "AREA":      f["area"] = AREA_ALIASES.get(e.text.lower())
        elif e.label_ == "PARTY":     f["parties"].add(e.text.lower())
    f["parties"] = sorted(f["parties"]); return f

### 📸 A. Clause splitting

In [ ]:
text = open("sample_agreement.txt", encoding="utf-8").read()
header, clauses = split_clauses(text)
print("Plain NLTK sent_tokenize on clause 1 ->", len(sent_tokenize(clauses[0])), "pieces  ❌ (splits at 'Rs.')")
print("Our splitter on the whole agreement   ->", len(clauses), "clauses ✅\n")
for i, c in enumerate(clauses, 1): print(f"{i}. {c}")
print("\nFallback (unnumbered text):", split_clauses("Rent is Rs. 30,000 per month. Deposit is Rs. 60,000. Mr. Rao is the Landlord.")[1])

### 📸 B. Tokens, POS tags, lemmas, dependencies, noun chunks

In [ ]:
doc = nlp_base(clauses[4])
display(pd.DataFrame([(t.text, t.pos_, t.lemma_, t.dep_) for t in doc], columns=["Token","POS","Lemma","Dependency"]).head(14))
print("Noun chunks:", [c.text for c in doc.noun_chunks])

### 📸 C. Named entities in the preamble (pretrained model only)

In [ ]:
pre = nlp_base(header.replace("LEAVE AND LICENSE AGREEMENT", "").strip())
display(pd.DataFrame([(e.text, e.label_, spacy.explain(e.label_)) for e in pre.ents], columns=["Entity","Label","Meaning"]))
print("Note: the small pretrained model makes mistakes here — one reason we add our own rules.")

### 📸 D. Pretrained NER vs our Hybrid NER (the key result of this step)

In [ ]:
c = clauses[1]; print(c, "\n")
print("Pretrained only:", [(e.text, e.label_) for e in nlp_base(c).ents])
print("Hybrid (rules) :", [(e.text, e.label_) for e in nlp(c).ents])

### 📸 E. Entities and extracted facts for every clause

In [ ]:
rows = []
for i, c in enumerate(clauses, 1):
    d = nlp(c)
    ents = ", ".join(f"{e.text} [{e.label_}]" for e in d.ents if e.label_ in
                     ("MONEY_AMT","PERCENT_VAL","DURATION","AREA","BHK"))
    facts = {k: v for k, v in extract_facts(c).items() if v and k != "parties"}
    rows.append((i, c[:55] + "...", ents, facts))
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows, columns=["#","Clause","Entities found","Extracted facts"])

## STEP 4 — Vectorization & Classification  *(Practical 3/4 → TF-IDF + ML models)*

**Goal:** teach a model to predict, for each clause, (a) its **type** (deposit, lock-in, eviction …) and (b) its **risk level** (Low / Medium / High).

**What happens here**
1. **Data augmentation** – each clause is rewritten in 3 other styles (*Licensee/Licensor*, *Lessee/Owner*, `₹` / `INR`, `%`) so the model sees varied wording. These are **synthetic**, and we say so in the report.
2. **TF-IDF with 1–2-word n-grams** – turns text into numbers. n-grams matter: *"without notice"* means something different from *"notice"* alone.
3. **Three models compared** – Naive Bayes, Logistic Regression, Linear SVM (+ a *dummy* baseline that always guesses the commonest class).
4. **Leak-proof evaluation** – `GroupKFold`: all rewrites of one clause stay in the *same* fold. Without this, the model would be tested on rewrites of clauses it already trained on and the score would be fake.
5. **A separate unseen test set** – 52 clauses written in *new wording* that never appear in training.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

# ---------- 4a. Training data = base clauses + 3 rewrites each ----------
base = pd.read_csv("clauses.csv"); base["group"] = range(len(base))

def make_variants(t):
    v1 = t.replace("Tenant", "Licensee").replace("Landlord", "Licensor")
    v2 = t.replace("shall", "will").replace("Rs. ", "₹")
    v3 = (t.replace("Tenant", "Lessee").replace("Landlord", "Owner")
            .replace(" percent", "%").replace("Rs. ", "INR "))
    return list(dict.fromkeys([t, v1, v2, v3]))          # unique, order kept

rows = [(v, r.clause_type, r.risk_level, r.group, r.label_source) for r in base.itertuples() for v in make_variants(r.clause_text)]
train = pd.DataFrame(rows, columns=["clause_text", "clause_type", "risk_level", "group", "label_source"])
train["clean_text"] = train.clause_text.apply(preprocess)
print(f"Base clauses: {len(base)}  →  training rows after augmentation: {len(train)}")
print("Wording-based (hand-labelled) rows used to train the RISK model:", (train.label_source == "hand").sum())
display(train.sample(5, random_state=3)[["clause_text", "clause_type", "risk_level"]])

### Unseen test set (new wording, never used for training)
Numeric clauses are labelled by the Pune rules; the rest are hand-labelled. ⚠️ I (the author) wrote and labelled these — in the report, call it a *held-out test set*, and also add real clauses in the optional cell further below.

In [ ]:
test_rows = []          # (text, type, risk, is_numeric)
def T(text, ctype, risk, numeric=False): test_rows.append((text, ctype, risk, numeric))

# ---- numeric clauses: risk label computed from Pune rules ----
for text, key, bhk, rent in [
    ("Licence fee for the 2BHK apartment at Wakad is fixed at ₹29,500 per month.", "wakad", 2, 29500),
    ("The Licensee shall pay INR 41,000 each month as rent for the 2 BHK unit in Kothrud.", "kothrud", 2, 41000),
    ("Monthly licence fee for the 1 BHK flat in Viman Nagar: Rs. 27,000.", "viman nagar", 1, 27000),
    ("The rent payable for the 3 BHK flat in Baner shall be ₹82,000 per month.", "baner/balewadi", 3, 82000),
    ("The Licensee agrees to a monthly rent of Rs. 26,000 for a 2 BHK in Sus Gaon.", "sus gaon", 2, 26000),
    ("The flat in Kharadi (1 BHK) shall be let at a rent of Rs. 36,000 per month.", "kharadi", 1, 36000)]:
    T(text, "rent", rent_risk(key, bhk, rent), True)

for text, rent, dep in [
    ("Against a monthly rent of Rs. 30,000, the Licensee shall deposit Rs. 60,000 as interest free refundable security.", 30000, 60000),
    ("The security deposit is Rs. 1,05,000, while the monthly rent is Rs. 35,000.", 35000, 105000),
    ("Interest-free security deposit of ₹2,50,000 shall be paid by the Licensee; the rent is ₹40,000 per month.", 40000, 250000),
    ("Licensee to pay deposit equal to Rs. 45,000 on a rent of Rs. 30,000.", 30000, 45000)]:
    T(text, "deposit", deposit_risk(dep, rent), True)

T("The Licensor shall return the deposit within 20 days after the Licensee hands over the keys.", "deposit", refund_risk(20), True)
T("Refund of the deposit will be made within ninety days from vacating the flat.", "deposit", refund_risk(90), True)

for text, p in [("Annual revision in rent shall not exceed 5%.", 5),
                ("At renewal, the licence fee will go up by ten percent.", 10),
                ("A hike of 12% in the monthly rent applies on every renewal.", 12),
                ("Rent escalation of 15 percent per annum shall apply.", 15),
                ("The Licensor may raise the rent by 20% every year.", 20),
                ("Every year the rent shall be increased by thirty percent.", 30)]:
    T(text, "rent_increase", increase_risk(p), True)

for text, grace, fee in [
    ("If rent is not received within 6 days of the due date, Rs. 150 per day is charged.", 6, 150),
    ("Delay in rent beyond 3 days attracts a fine of ₹300 per day.", 3, 300),
    ("A late charge of Rs. 1,000 per day applies immediately after the due date, no grace is given.", 0, 1000),
    ("Late payment fee is Rs. 200 per day after a grace period of five days.", 5, 200)]:
    T(text, "penalty", latefee_risk(grace, fee), True)

# ---- wording-based clauses: hand-labelled ----
for text, ctype, risk in [
 ("The Licensor can keep any part of the security deposit as he pleases, without showing bills.", "deposit", "High"),
 ("Deductions from the deposit, if any, will be supported by receipts of actual repair costs.", "deposit", "Low"),
 ("The deposit will be forfeited in full if the Licensee gives notice before the end of the agreement period.", "deposit", "High"),
 ("A fine will be levied on the Licensee whenever the Licensor feels a rule has been broken.", "penalty", "High"),
 ("If the Licensee is late with rent, no charge will apply for the first week.", "penalty", "Low"),
 ("Using the parking slot of another resident will attract a fixed fine of three months' rent.", "penalty", "High"),
 ("Neither party can end the agreement in the first four months; after that one month's notice is enough.", "lockin", "Low"),
 ("The Licensee is bound to stay for the whole eleven months and must pay all remaining rent if he leaves early.", "lockin", "High"),
 ("A ten month commitment applies to the Licensee alone; the Licensor is free to end the arrangement whenever he wants.", "lockin", "High"),
 ("Both parties agree to a minimum stay of six months, with a penalty of one month's rent for leaving early.", "lockin", "Low"),
 ("After the first seven months, either side may leave by giving thirty days' written notice.", "lockin", "Low"),
 ("The Licensee may leave early only if the Licensor agrees, and the Licensor can refuse without any reason.", "lockin", "Medium"),
 ("The Licensor shall give ninety days' written notice before asking the Licensee to vacate for any reason other than breach.", "eviction", "Low"),
 ("The Licensor may throw out the Licensee at once if rent is late, with no chance to pay.", "eviction", "High"),
 ("Termination by the Licensor needs a written notice of one month and a valid reason.", "eviction", "Low"),
 ("The Licensee must hand over the flat within two days of the Licensor's oral instruction.", "eviction", "High"),
 ("If the owner wants the flat for family use, the Licensee shall be given twenty days' notice.", "eviction", "Medium"),
 ("On repeated non-payment, the Licensor may end the licence after giving a fortnight's notice.", "eviction", "Medium"),
 ("Society dues, including sinking fund and corpus contributions, will be borne by the owner.", "maintenance", "Low"),
 ("All repair work, big or small, is the responsibility of the Licensee.", "maintenance", "High"),
 ("The Licensee pays for painting of the flat when leaving, whatever its condition.", "maintenance", "Medium"),
 ("Maintenance charges will be paid by the Licensee on production of the society's latest bill.", "maintenance", "Medium"),
 ("The owner shall fix leakage and wiring issues within a week of being informed.", "maintenance", "Low"),
 ("Charges for the clubhouse and shifting in and out will be split between both parties.", "maintenance", "Medium"),
 ("The Licensee will pay the actual electricity bill generated by the meter.", "utilities", "Low"),
 ("Power will be billed to the Licensee at a rate set by the Licensor, higher than the official tariff.", "utilities", "High"),
 ("Water charges are Rs. 1,200 monthly, fixed, however much water is used.", "utilities", "Medium"),
 ("Gas and broadband bills are the responsibility of the Licensee.", "utilities", "Low"),
 ("The Licensee shall also reimburse the owner for the annual property tax of the building.", "utilities", "High"),
 ("The Licensor may stop water and power if payment is delayed even slightly.", "utilities", "High")]:
    T(text, ctype, risk, False)

test = pd.DataFrame(test_rows, columns=["clause_text", "clause_type", "risk_level", "numeric"])
test["clean_text"] = test.clause_text.apply(preprocess)
print("Test clauses:", len(test), "| numeric:", test.numeric.sum(), "| wording-based:", (~test.numeric).sum())
display(pd.crosstab(test.clause_type, test.risk_level))

### 📸 4b. Model comparison with 5-fold grouped cross-validation (training data only)

In [ ]:
def build(model):
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)), ("clf", model)])

factories = {
    "Baseline (most frequent)": lambda: DummyClassifier(strategy="most_frequent"),
    "Naive Bayes":              lambda: MultinomialNB(alpha=0.3),
    "Logistic Regression":      lambda: LogisticRegression(max_iter=3000, C=10),
    "Linear SVM":               lambda: LinearSVC(C=1),
}
# The RISK model only learns from wording-based clauses ("hand"): numeric clauses are judged by Pune rules (Step 5).
TASKS = {"clause_type": ("Clause TYPE", None), "risk_level": ("RISK (wording clauses)", "hand")}
cv_rows = []
for task, (tname, src) in TASKS.items():
    data = train if src is None else train[train.label_source == src]
    for name, make in factories.items():
        r = cross_validate(build(make()), data.clean_text, data[task], groups=data.group,
                           cv=GroupKFold(n_splits=5), scoring=["accuracy", "f1_macro"])
        cv_rows.append((tname, name, r["test_accuracy"].mean(), r["test_f1_macro"].mean(), r["test_f1_macro"].std()))
cv = pd.DataFrame(cv_rows, columns=["Task", "Model", "CV Accuracy", "CV Macro-F1", "F1 std-dev"]).round(3)
display(cv)

### 📸 4c. Final test on UNSEEN clauses (models trained on all training data)

In [ ]:
fitted, best, test_rows_out = {}, {}, []
def subsets(task):
    src = TASKS[task][1]
    return (train if src is None else train[train.label_source == src]), (test if src is None else test[~test.numeric])

for task, (tname, src) in TASKS.items():
    fitted[task] = {}; tr, te = subsets(task)
    for name, make in factories.items():
        m = build(make()).fit(tr.clean_text, tr[task]); fitted[task][name] = m
        p = m.predict(te.clean_text)
        test_rows_out.append((tname, name, accuracy_score(te[task], p),
                              f1_score(te[task], p, average="macro"),
                              f1_score(te[task], p, average="weighted")))
    # best real model chosen by cross-validation Macro-F1 (never by the test set)
    sub = cv[(cv.Task == tname) & (~cv.Model.str.startswith("Baseline"))]
    best[task] = sub.sort_values("CV Macro-F1", ascending=False).iloc[0].Model

res = pd.DataFrame(test_rows_out, columns=["Task", "Model", "Test Accuracy", "Test Macro-F1", "Test Weighted-F1"]).round(3)
display(res)
print("Best model chosen by cross-validation →", {TASKS[k][0]: v for k, v in best.items()})

### 📸 4d. Precision / Recall / F1 and confusion matrices of the best models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (task, (tname, src)) in zip(axes, TASKS.items()):
    te = subsets(task)[1]
    labels = ["Low", "Medium", "High"] if task == "risk_level" else sorted(test[task].unique())
    pred = fitted[task][best[task]].predict(te.clean_text)
    print(f"\n===== {tname} — {best[task]} (on {len(te)} unseen clauses) =====")
    print(classification_report(te[task], pred, labels=labels, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(te[task], pred, labels=labels, ax=ax,
                                            cmap="Blues", xticks_rotation=45, colorbar=False)
    ax.set_title(f"{tname}: {best[task]}")
plt.tight_layout(); plt.show()

### 📸 4e. Why ML alone is not enough for risk — the key finding
Our **ML-only baseline** is the same best model trained on *every* clause (numeric ones too) with no rules. A text model can recognise *wording* like "without notice", but it **cannot do arithmetic**: "deposit of Rs. 1,05,000 on rent of Rs. 35,000" needs a ratio (3×), and "Rs. 41,000 for a 2 BHK in Kothrud" needs the local market range. Let's measure it.

In [ ]:
ml_all = build(factories[best["risk_level"]]()).fit(train.clean_text, train.risk_level)   # ML-only baseline
test["ml_risk"] = ml_all.predict(test.clean_text)
out = pd.DataFrame({
    "Subset": ["Wording-based clauses", "Numeric clauses (rent, deposit, %, late fee)", "All clauses"],
    "Clauses": [(~test.numeric).sum(), test.numeric.sum(), len(test)],
    "ML-only risk accuracy": [
        accuracy_score(test[~test.numeric].risk_level, test[~test.numeric].ml_risk),
        accuracy_score(test[test.numeric].risk_level,  test[test.numeric].ml_risk),
        accuracy_score(test.risk_level, test.ml_risk)]}).round(3)
display(out)

## STEP 5 — Hybrid Risk Engine with Explanations

**Design:** for every clause
1. spaCy hybrid NER finds the facts (rent, deposit, %, days, area, BHK).
2. **If a Pune rule applies** (numbers present) → the rule decides the risk and writes the reason, e.g. *"Deposit is 3.5× the monthly rent"*.
3. **Otherwise** → the ML model decides from wording, and we show the **top words** that pushed the decision (from Logistic Regression coefficients).
4. The **clause type** comes from the rule that fired (it knows the family: rent, deposit, increase, late fee); otherwise from the ML model.
5. All clauses are combined into an **overall risk score**: each **High** clause adds 15 points and each **Medium** 7 (capped at 100). Below 25 = LOW, below 60 = MODERATE, otherwise HIGH. *(A plain average hid risky clauses among ordinary boilerplate, so I switched to this penalty score — the weights are a design choice, stated in the report.)*

`ctx` remembers the rent found in an earlier clause, so a deposit clause that says only *"deposit of Rs. 1,26,000"* can still be compared with the rent.

In [ ]:
type_model = fitted["clause_type"][best["clause_type"]]
risk_model = fitted["risk_level"][best["risk_level"]]
lr_risk    = fitted["risk_level"]["Logistic Regression"]       # used only to EXPLAIN word-based decisions

def top_words(clean, label, n=3):
    vec, clf = lr_risk.named_steps["tfidf"], lr_risk.named_steps["clf"]
    contrib = vec.transform([clean]).multiply(clf.coef_[list(clf.classes_).index(label)]).toarray()[0]
    names = vec.get_feature_names_out()
    return [names[i] for i in contrib.argsort()[::-1][:n] if contrib[i] > 0]

def rule_based_risk(clause, ctx):
    """Returns (risk, reason, family) if a Pune rule applies, else (None, None, None)."""
    d, low = nlp(clause), clause.lower()
    ents  = [(e.label_, e.text, e.start_char) for e in d.ents]
    money = [(money_value(t), s) for l, t, s in ents if l == "MONEY_AMT"]
    pcts  = [first_number(t) for l, t, s in ents if l == "PERCENT_VAL"]
    days  = [duration_days(t) for l, t, s in ents if l == "DURATION" and "day" in t.lower()]
    area  = next((AREA_ALIASES.get(t.lower()) for l, t, s in ents if l == "AREA"), None)
    bhk   = next((int(first_number(t)) for l, t, s in ents if l == "BHK"), None)

    if "deposit" in low:                                            # --- deposit refund time
        if days and re.search(r"refund|return", low) and not money:
            return refund_risk(days[0]), f"Deposit refunded after {days[0]} days (Pune norm: 15–30 days)", "deposit"
        if money:                                                   # --- deposit vs rent
            k = low.find("deposit")
            after = [m for m, s in money if s > k]
            dep = after[0] if after else money[0][0]
            others = [m for m, s in money if m != dep]
            rent = others[0] if others else ctx.get("rent")
            if rent:
                return deposit_risk(dep, rent), f"Deposit is {dep/rent:.1f}x monthly rent (Low ≤2x, Medium ≤3x, High >3x)", "deposit"
        return None, None, None
    if pcts and re.search(r"increas|revis|escalat|hike|raise|go up|goes up", low):    # --- yearly increase
        return increase_risk(pcts[0]), f"Rent rises {pcts[0]:g}% per renewal (Low ≤10%, Medium ≤15%, High >15%)", "rent_increase"
    if money and re.search(r"\blate\b|delay|overdue|due date", low) and re.search(r"per day|a day|each day|daily", low):
        grace = days[0] if days else 0                              # --- late fee
        return latefee_risk(grace, money[0][0]), f"Late fee Rs.{money[0][0]}/day with {grace}-day grace (norm: modest fee after 5–7 days)", "penalty"
    if area and bhk in (1, 2, 3) and money:                         # --- rent vs market
        rent = money[0][0]; lo, hi = RENT_RANGES[area][bhk]; ctx.update(rent=rent, area=area, bhk=bhk)
        return rent_risk(area, bhk, rent), f"Rent Rs.{rent:,} is Rs.{max(rent-lo,0):,} above the start of the Pune {area.title()} {bhk} BHK range (Rs.{lo:,}–{hi:,}); Low up to +4,000, Medium up to +7,000", "rent"
    return None, None, None

def hybrid_predict(clause, ctx):
    clean = preprocess(clause)
    ctype = type_model.predict([clean])[0]
    risk, reason, family = rule_based_risk(clause, ctx)
    if risk is not None:
        return family, risk, "Pune rule", reason
    risk = risk_model.predict([clean])[0]
    words = top_words(clean, risk)
    if risk == "Low": return ctype, risk, "ML (wording)", "No risky terms found - standard wording"
    return ctype, risk, "ML (wording)", "Key phrases: " + (", ".join(words) if words else "overall wording pattern")

### 📸 5a. Does the hybrid beat ML alone? (same unseen test set)

In [ ]:
preds = [hybrid_predict(c, {}) for c in test.clause_text]
test["hy_type"], test["hy_risk"], test["source"], test["reason"] = zip(*preds)

def acc(df, col): return accuracy_score(df.risk_level, df[col])
cmp_df = pd.DataFrame({
    "Subset": ["Wording-based", "Numeric", "All clauses"],
    "ML only": [acc(test[~test.numeric], "ml_risk"), acc(test[test.numeric], "ml_risk"), acc(test, "ml_risk")],
    "Hybrid (rules + ML)": [acc(test[~test.numeric], "hy_risk"), acc(test[test.numeric], "hy_risk"), acc(test, "hy_risk")]}).round(3)
display(cmp_df)
print(f"Overall risk Macro-F1:  ML only = {f1_score(test.risk_level, test.ml_risk, average='macro'):.3f}"
      f"  |  Hybrid = {f1_score(test.risk_level, test.hy_risk, average='macro'):.3f}")
ml_type = type_model.predict(test.clean_text)
print(f"Clause-TYPE accuracy:  ML only = {accuracy_score(test.clause_type, ml_type):.3f}  |  Hybrid (rule family when a rule fires) = {accuracy_score(test.clause_type, test.hy_type):.3f}")

ax = cmp_df.set_index("Subset").plot(kind="bar", figsize=(7, 4), color=["#90a4ae", "#1e88e5"])
ax.set_ylim(0, 1.05); ax.set_ylabel("Risk accuracy"); ax.set_title("ML only vs Hybrid on unseen clauses")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### 📸 5b. Error analysis — every clause the hybrid got wrong (use this in the report & viva)

In [ ]:
wrong = test[test.risk_level != test.hy_risk][["clause_text", "risk_level", "hy_risk", "source", "reason"]]
wrong.columns = ["Clause", "True risk", "Predicted", "Decided by", "Reason shown"]
pd.set_option("display.max_colwidth", 90)
print(f"{len(wrong)} of {len(test)} risk predictions wrong")
display(wrong)

### 📸 5c. Full analysis of the sample agreement

In [ ]:
def risk_score(risks):
    score = min(100, sum(15 if r == "High" else 7 if r == "Medium" else 0 for r in risks))
    return score, ("LOW" if score < 25 else "MODERATE" if score < 60 else "HIGH")

def analyse_agreement(text):
    header, clauses = split_clauses(text)
    ctx, out = {}, []
    for i, c in enumerate(clauses, 1):
        ctype, risk, source, reason = hybrid_predict(c, ctx)
        out.append((i, c, ctype, risk, source, reason))
    rep = pd.DataFrame(out, columns=["#", "Clause", "Type", "Risk", "Decided by", "Why"])
    score, _ = risk_score(rep.Risk)
    return rep, score

COL = {"Low": "background-color:#c8e6c9;color:#1b1b1b;font-weight:600", "Medium": "background-color:#fff3b0;color:#1b1b1b;font-weight:600", "High": "background-color:#ffcdd2;color:#1b1b1b;font-weight:600"}
report, score = analyse_agreement(open("sample_agreement.txt", encoding="utf-8").read())
score, level = risk_score(report.Risk)
display(report.style.apply(lambda col: [COL[v] for v in col], subset=["Risk"])
        .set_properties(subset=["Clause", "Why"], **{"text-align": "left", "white-space": "normal"}))
print(f"\nOVERALL RISK SCORE: {score}/100  →  {level} RISK")
print("High-risk clauses:", list(report[report.Risk == "High"]["#"]), "| Medium:", list(report[report.Risk == "Medium"]["#"]))

### Optional (strongly recommended): test on REAL clauses
Upload a CSV named **`real_clauses.csv`** (Colab left panel ▸ 📁 Files ▸ Upload) with columns `clause_text, clause_type, risk_level`, using 20–30 clauses from real agreements (remove names). The cell below then evaluates the system on them. If you have none yet, it simply skips.

In [ ]:
import os
if os.path.exists("real_clauses.csv"):
    real = pd.read_csv("real_clauses.csv")
    preds = [hybrid_predict(c, {}) for c in real.clause_text]
    real["pred_type"], real["pred_risk"], real["source"], real["reason"] = zip(*preds)
    print(f"Real clauses: {len(real)}")
    print(f"Type accuracy: {accuracy_score(real.clause_type, real.pred_type):.3f} | "
          f"Risk accuracy: {accuracy_score(real.risk_level, real.pred_risk):.3f} | "
          f"Risk macro-F1: {f1_score(real.risk_level, real.pred_risk, average='macro'):.3f}")
    display(real[real.risk_level != real.pred_risk][["clause_text", "risk_level", "pred_risk", "reason"]])
else:
    print("real_clauses.csv not found — skipping (add it to report results on real-world text).")

## STEP 6 — Official Maharashtra Form Mode *(live demo)*

The real **Leave and License Agreement** format (Maharashtra IGR) differs from my training wording: it says *Licensor/Licensee* and *"compensation"* (not rent), puts the rent **and** the deposit in one clause (Clause 2), has **staged-rent** and **non-refundable-deposit** options, and has no lock-in, increase or late-fee clause at all.

**Upgrades in this step**
1. **Money-role detection** – decides whether each `Rs.` amount is rent or deposit from the words around it (e.g. *"Rs. 36,000 per month"* vs *"Rs. 72,000 interest free refundable deposit"*).
2. **Clause 2 composite check** – one clause, several risks: rent vs Pune market range, deposit ÷ rent, non-refundable deposit, and **staged-rent step-ups** (checked with the yearly-increase rule).
3. **Lock-in rule** (mutual vs one-sided) and a transparent **red-flag phrase lexicon** (e.g. *"without recourse to the Court"*, *"double the daily amount"*).
4. **Form splitter** for the official `1) Title:` layout.
5. **Clause titles** come from the form's own headings (Period, Rent & Deposit, Possession …), which is more reliable than guessing the type.
6. **Fill-the-form panel** — you enter the rates, the notebook writes the agreement using the official clause wording, and analyses it.

In [ ]:
import re
from pune_rules import lockin_risk

# ---------- 6a. Official clause wording (Government of Maharashtra, IGR format) + common add-ons ----------
OFFICIAL = {
 "period": "That the Licensor hereby grants to the Licensee herein a revocable leave and license, to occupy the Licensed Premises, described in Schedule hereunder written without creating any tenancy rights or any other rights, title and interest in favour of the Licensee for a period of {months} Months.",
 "deposit_pay": "That the Licensee shall pay the above mentioned deposit to the Licensor by bank transfer on signing this agreement.",
 "maint_licensor": "That the all outgoings including all rates, taxes, levies, assessment, maintenance charges, non occupancy charges, etc. in respect of the said premises shall be paid by the Licensor.",
 "maint_licensee": "That the Licensee/s herein shall bear and pay all the maintenance charges in respect of the said Licensed Premises, and other outgoings including all rates, taxes, levies, assessment, non occupancy charges, etc. in respect of the said premises shall be paid by the Licensor/s.",
 "use": "That the Licensed premises shall only be used by the Licensee for residential purpose. The Licensee shall maintain the said premises in its existing condition and damage, if any, caused to the said premises, the same shall be repaired by the Licensee at its own cost subject to normal wear and tear. The Licensee shall not do anything in the said premises which is or is likely to cause a nuisance to the other occupants of the said building or to the prejudice in any manner to the rights of Licensor in respect of said premises or shall not do any unlawful activities prohibited by State or Central Government.",
 "alteration": "That the Licensee shall not make or permit to do any alteration or addition to the construction or arrangements (internal or external) to the Licensed premises without previous consent in writing from the Licensor.",
 "no_tenancy": "That the Licensee shall not claim any tenancy right and shall not have any right to transfer, assign, and sublet or grant any license or sub-license in respect of the Licensed Premises or any part thereof and also shall not mortgage or raise any loan against the said premises.",
 "inspection": "That, the Licensor shall on reasonable notice given by the Licensor to the Licensee shall have a right of access either by himself or through authorized representative to enter, view and inspect the Licensed premises at reasonable intervals.",
 "cancellation": "That, if the Licensee commits default in regular and punctual payments of monthly compensation as herein before mentioned or commit/s breach of any of the terms, covenants and conditions of this agreement or if any legislation prohibiting the Leave and License is imposed, the Licensor shall be entitled to revoke and / or cancel the License hereby granted, by giving notice in writing of one month and the Licensee too will have the right to vacate the said premises by giving a notice in writing of one month to the Licensor as mentioned earlier.",
 "possession": "That the immediately on the expiration or termination or cancellation of this agreement the Licensee shall vacate the said premises without delay with all his goods and belongings. In the event of the Licensee failing and / or neglecting to remove himself and / or his articles from the said premises on expiry or sooner determination of this Agreement, the Licensor shall be entitled to recover damages at the rate of double the daily amount of compensation per day and or alternatively the Licensor shall be entitled to remove the Licensee and his belongings from the Licensed premises, without recourse to the Court of Law.",
 "registration": "This Agreement is to be registered and the expenditure of Stamp duty and registration fees and incidental charges, if any, shall be borne by the Licensee.",
}

def indian(n):
    s = str(int(n))
    if len(s) <= 3: return s
    head, tail, parts = s[:-3], s[-3:], []
    while len(head) > 2: parts.insert(0, head[-2:]); head = head[:-2]
    if head: parts.insert(0, head)
    return ",".join(parts + [tail])

def build_agreement(area_name, bhk, months, mode, rent, rent2, m1, dep, deptype, maint,
                    lock, lockwho, esc, fee, grace, refund):
    """Writes the agreement using the official clause wording + the values typed into the form."""
    dtxt = "nonrefundable" if deptype == "Non-refundable" else "refundable"
    if mode == "Staged rent":
        rent_clause = (f"That the Licensee shall pay to the Licensor the following amount per month towards the compensation for the use of the said licensed premises. "
                       f"a) Rs. {indian(rent)} per month for the first {m1} months, b) Rs. {indian(rent2)} per month for the next {max(months-m1,1)} months. "
                       f"The amount of monthly compensation shall be payable within first five days of the concerned month of Leave and License. "
                       f"Licensee shall also pay to the Licensor Rs. {indian(dep)} interest free {dtxt} deposit, for the use of the said licensed premises.")
    else:
        rent_clause = (f"That the Licensee shall pay to the Licensor Rs. {indian(rent)} per month towards the compensation and Rs. {indian(dep)} interest free {dtxt} deposit, "
                       f"for the use of the said licensed premises. The amount of monthly compensation shall be payable within first five days of the concerned month of Leave and License.")
    items = [("Period", OFFICIAL["period"].format(months=months)), ("Rent & Deposit", rent_clause),
             ("Payment of Deposit", OFFICIAL["deposit_pay"]),
             ("Maintenance Charges", OFFICIAL["maint_licensee" if maint == "licensee" else "maint_licensor"]),
             ("Use", OFFICIAL["use"]), ("Alteration", OFFICIAL["alteration"]), ("No Tenancy", OFFICIAL["no_tenancy"]),
             ("Inspection", OFFICIAL["inspection"]), ("Cancellation", OFFICIAL["cancellation"]),
             ("Possession", OFFICIAL["possession"]), ("Registration", OFFICIAL["registration"])]
    # ---- add-ons that brokers/owners commonly insert (NOT in the official format) ----
    if lock:  items.append(("Lock-in", f"That there shall be a lock-in period of {lock} months " + ("applicable to both the Licensor and the Licensee." if lockwho == "both" else "applicable only to the Licensee, while the Licensor may terminate at any time.")))
    if esc:   items.append(("Increase", f"That the monthly compensation shall be increased by {esc:g} percent on renewal of the agreement after every eleven months."))
    if fee:   items.append(("Late Fee", f"That a late fee of Rs. {fee} per day shall be charged if the monthly compensation is not paid " + (f"within {grace} days of the due date." if grace else "from the day after the due date, with no grace period.")))
    if refund: items.append(("Deposit Refund", f"That the Licensor shall refund the deposit within {refund} days of handover of the premises."))
    body = "\n".join(f"{i}) {t}: {txt}" for i, (t, txt) in enumerate(items, 1))
    return ("LEAVE AND LICENSE AGREEMENT\nThis agreement is made at Pune between the Licensor and the Licensee (names withheld).\n\n" + body +
            f"\n\nSchedule: residential {bhk} BHK unit situated at {area_name}, Pune.\nIN WITNESS WHEREOF the parties have signed.")

print(build_agreement("Kothrud", 2, 11, "Fixed rent", 36000, 0, 6, 72000, "Refundable", "licensor", 6, "both", 5, 100, 7, 30)[:700], "...")

### 6b. Engine v2 — money roles, composite Clause-2 check, lock-in rule, red-flag lexicon

In [ ]:
STAGE_RE = re.compile(r"Rs\.?\s*([\d,]+)\s*per month for the (?:first|next)\s*(\d+)\s*months?", re.I)
RISK_RANK = {"Low": 0, "Medium": 1, "High": 2}
worst = lambda risks: max(risks, key=RISK_RANK.get)

RED_FLAGS = [   # (regex, explanation)  -> High
 (r"without recourse to (the )?court", "Licensor may remove the licensee without going to court"),
 (r"double (the )?(daily )?(amount|rate|rent|compensation)", "Penalty of double the daily compensation"),
 (r"sole discretion", "Decision left entirely to the Licensor"),
 (r"without (any )?(prior )?notice", "Action allowed without notice"),
 (r"forfeit|remaining rent|balance rent", "Deposit forfeited / remaining rent claimed"),
 (r"non[\s-]?refundable", "Amount cannot be recovered"),
]
AMBER_FLAGS = [  # -> Medium
 (r"licensee\S* herein shall bear and pay all the maintenance", "Licensee pays all maintenance (Pune norm: Licensor pays society maintenance)"),
 (r"(stamp duty|registration fee).{0,80}borne by the licensee", "Registration costs placed on the Licensee"),
]

def classify_money(clause, s, e):
    """Is this Rs. amount a monthly rent or a deposit? Uses the words around it."""
    low = clause.lower(); before = low[max(0, s - 80):s]; after = low[e:e + 45]
    m = re.search(r"(rent|compensation|licen[cs]e fee|deposit)(?:\s+(?:is|are|of|equal to|at|fixed at|fixed|shall be|will be|be))*\s*:?\s*$", before)
    if m: return "deposit" if m.group(1) == "deposit" else "rent"          # 1. cue right before the amount
    m = re.search(r"per month|monthly|a month|each month|every month|deposit", after)
    if m: return "deposit" if m.group() == "deposit" else "rent"           # 2. cue right after the amount
    last = None
    for mt in re.finditer(r"deposit|rent|compensation|licen[cs]e fee|monthly", before): last = mt.group()
    return None if last is None else ("deposit" if last == "deposit" else "rent")   # 3. nearest cue before

def rule_engine_v2(clause, ctx):
    """Returns (risk, reason, family, source) or (None, None, None, None)."""
    d, low = nlp(clause), clause.lower()
    ents  = [(e.label_, e.text, e.start_char, e.end_char) for e in d.ents]
    money = [(money_value(t), s, e) for l, t, s, e in ents if l == "MONEY_AMT"]
    pcts  = [first_number(t) for l, t, s, e in ents if l == "PERCENT_VAL"]
    days  = [duration_days(t) for l, t, s, e in ents if l == "DURATION" and "day" in t.lower()]
    mons  = [duration_days(t) / 30 for l, t, s, e in ents if l == "DURATION" and "month" in t.lower()]
    area  = next((AREA_ALIASES.get(t.lower()) for l, t, s, e in ents if l == "AREA"), None) or ctx.get("area")
    bhk   = next((int(first_number(t)) for l, t, s, e in ents if l == "BHK"), None) or ctx.get("bhk")

    # 1. late fee
    if money and re.search(r"\blate\b|delay|overdue|due date", low) and re.search(r"per day|a day|each day|daily", low):
        grace = days[0] if days else 0
        return latefee_risk(grace, money[0][0]), f"Late fee Rs.{money[0][0]}/day with {grace}-day grace (norm: modest fee after 5-7 days)", "penalty", "Pune rule"
    # 2. yearly increase
    if pcts and re.search(r"increas|revis|escalat|hike|raise|go up|goes up", low):
        return increase_risk(pcts[0]), f"Rent rises {pcts[0]:g}% per renewal (Low <=10%, Medium <=15%, High >15%)", "rent_increase", "Pune rule"
    # 3. rent / deposit (Clause 2 of the official format)
    if "deposit" in low or re.search(r"rent|compensation|licen[cs]e fee", low):
        parts, fam = [], set()
        if "deposit" in low and days and re.search(r"refund|return", low) and not money:
            return refund_risk(days[0]), f"Deposit refunded after {days[0]} days (Pune norm: 15-30 days)", "deposit", "Pune rule"
        roles = [(v, classify_money(clause, s, e)) for v, s, e in money]
        stages = [(money_value(a), int(b)) for a, b in STAGE_RE.findall(clause)]
        rents = [v for v, r in roles if r == "rent"]; deps = [v for v, r in roles if r == "deposit"]
        rent_ref = stages[0][0] if stages else (rents[0] if rents else ctx.get("rent"))
        if (stages or rents) and area and bhk in (1, 2, 3):
            lo, hi = RENT_RANGES[area][bhk]; fam.add("rent")
            for k, r in enumerate(([x[0] for x in stages] if stages else rents[:1]), 1):
                lab = f"Stage {k} " if stages else ""
                parts.append((rent_risk(area, bhk, r), f"{lab}rent Rs.{r:,} is Rs.{max(r-lo,0):,} above the start of the Pune {area.title()} {bhk} BHK range (Rs.{lo:,}-{hi:,}); Low up to +4,000, Medium up to +7,000"))
            if stages:
                for (r1, _), (r2, _) in zip(stages, stages[1:]):
                    if r1 and r2 > r1:
                        p = (r2 - r1) / r1 * 100; parts.append((increase_risk(p), f"Staged rent steps up {p:.1f}% (norm: <=10% Low, <=15% Medium)"))
            ctx["rent"] = rent_ref
        if deps:
            fam.add("deposit"); dep = deps[0]
            if rent_ref: parts.append((deposit_risk(dep, rent_ref), f"Deposit is {dep/rent_ref:.1f}x monthly rent (Low <=2x, Medium <=3x, High >3x)"))
            if re.search(r"non[\s-]?refundable", low): parts.append(("High", "Deposit is NON-refundable - tenant cannot recover it"))
        if parts:
            family = "rent_and_deposit" if len(fam) == 2 else fam.pop()
            return worst([p[0] for p in parts]), " | ".join(p[1] for p in parts), family, "Pune rule"
    # 4. lock-in
    if re.search(r"lock-?in", low) and mons:
        one = bool(re.search(r"only to the licensee|licensee only|licensee alone|only for the licensee|applicable only", low))
        flag = next((why for pat, why in RED_FLAGS if re.search(pat, low)), None)
        if not flag:
            return lockin_risk(mons[0], one), f"Lock-in {mons[0]:g} months, {'one-sided (Licensee only)' if one else 'mutual'} (Low <=6, Medium <=11, High >11 or one-sided)", "lockin", "Pune rule"
    # 5. red-flag / amber phrase lexicon
    for pat, why in RED_FLAGS:
        if re.search(pat, low): return "High", "Red-flag phrase: " + why, None, "Red-flag lexicon"
    for pat, why in AMBER_FLAGS:
        if re.search(pat, low): return "Medium", "Amber phrase: " + why, None, "Red-flag lexicon"
    return None, None, None, None

def hybrid_predict_v2(clause, ctx, rmodel=None):
    rmodel = rmodel or risk_model
    clean = preprocess(clause); ctype = type_model.predict([clean])[0]
    risk, reason, family, source = rule_engine_v2(clause, ctx)
    if risk is not None: return (family or ctype), risk, source, reason
    risk = rmodel.predict([clean])[0]; words = top_words(clean, risk)
    if risk == "Low": return ctype, risk, "ML (wording)", "No risky terms found - standard wording"
    return ctype, risk, "ML (wording)", "Key phrases: " + (", ".join(words) if words else "overall wording pattern")

### 📸 6c. Regression check — did the upgrade break anything? (same 52 unseen clauses as Step 5)

In [ ]:
v2 = [hybrid_predict_v2(c, {}) for c in test.clause_text]
test["v2_risk"] = [p[1] for p in v2]
chk = pd.DataFrame({"Subset": ["Wording-based", "Numeric", "All clauses"],
    "Step 5 hybrid": [acc(test[~test.numeric], "hy_risk"), acc(test[test.numeric], "hy_risk"), acc(test, "hy_risk")],
    "Step 6 engine v2": [acc(test[~test.numeric], "v2_risk"), acc(test[test.numeric], "v2_risk"), acc(test, "v2_risk")]}).round(3)
display(chk)
print("Macro-F1 (risk):  Step 5 =", round(f1_score(test.risk_level, test.hy_risk, average="macro"), 3),
      " | Step 6 =", round(f1_score(test.risk_level, test.v2_risk, average="macro"), 3))

### 📸 6d. Check on the REAL wording of the official form
The 10 non-numeric clauses of the official Maharashtra format, labelled by me (Low / Medium / High). ⚠️ Honest caveat: I read this form while writing the red-flag lexicon, so this is a **development check, not an independent test**. The independent test is your **classmate-labelled / real clauses** (see the optional cell in Step 5). After measuring, the official clauses are added to training for the deployed model — the usual practice of training the final model on all labelled data.

In [ ]:
official_labelled = [
 (OFFICIAL["period"].format(months=11), "Low"),   (OFFICIAL["maint_licensor"], "Low"),
 (OFFICIAL["maint_licensee"], "Medium"),          (OFFICIAL["use"], "Low"),
 (OFFICIAL["alteration"], "Low"),                 (OFFICIAL["no_tenancy"], "Low"),
 (OFFICIAL["inspection"], "Low"),                 (OFFICIAL["cancellation"], "Low"),
 (OFFICIAL["possession"], "High"),                (OFFICIAL["registration"], "Medium")]
of = pd.DataFrame(official_labelled, columns=["clause_text", "risk_level"])
of["ml_only"]  = risk_model.predict(of.clause_text.apply(preprocess))
of["engine_v2"] = [hybrid_predict_v2(c, {})[1] for c in of.clause_text]
print(f"Official clauses (n={len(of)}):  ML-only accuracy = {accuracy_score(of.risk_level, of.ml_only):.2f} | "
      f"Engine v2 accuracy = {accuracy_score(of.risk_level, of.engine_v2):.2f}")
of_show = of.assign(clause=of.clause_text.str[:70] + "...")[["clause", "risk_level", "ml_only", "engine_v2"]]
of_show.columns = ["Clause", "My label", "ML only", "Engine v2"]; display(of_show)

# Deployed model = best wording model re-trained on wording clauses + the labelled official clauses
extra = pd.DataFrame([(v, r) for t, r in official_labelled for v in make_variants(t)], columns=["clause_text", "risk_level"])
wording = train[train.label_source == "hand"][["clause_text", "risk_level"]]
alld = pd.concat([wording, extra]); alld["clean"] = alld.clause_text.apply(preprocess)
final_risk_model = build(factories[best["risk_level"]]()).fit(alld.clean, alld.risk_level)
print("Deployed risk model re-trained on", len(alld), "clauses.")

### 6e. Analyser for filled official forms
`split_official()` understands the official layout (`1) Title: text`) and drops the schedule and signature block. `analyse_official()` returns the colour-coded report, an overall score and **"not stated"** warnings for protections the official format lacks.

In [ ]:
def split_official(text):
    t = re.sub(r"(\d)\s*/-", r"\1", text)                       # 36,000/-  -> 36,000
    t = re.split(r"\n\s*(?:Schedule:|IN WITNESS)", t)[0]          # drop schedule + signatures
    t = re.sub(r"LEAVE AND LICENSE AGREEMENT", " ", t)
    t = re.sub(r"Note:\s*The clause \d+ will depend on your selection", " ", t)
    out = []
    for p in re.split(r"(?m)^\s*\d+\)\s*", t)[1:]:
        p = re.sub(r"\s+", " ", p).strip()
        p = re.sub(r"\bOR\b\s*$", "", p).strip()
        hm = re.match(r"^([A-Za-z &/-]+):\s*", p)                   # heading such as 'Rent & Deposit:'
        head = hm.group(1).strip() if hm else "Clause"
        if hm: p = p[hm.end():]
        if p: out.append((head, p))
    return out

CHECKS = [("Lock-in period", r"lock-?in", "Without it either side may be able to leave or evict at will"),
          ("Yearly rent increase", r"increas|escalat|revision", "Without a cap the rent can rise by any amount on renewal"),
          ("Late-fee terms", r"late (fee|charge|payment)", "No stated grace period or penalty limit"),
          ("Deposit refund timeline", r"refund[^.]{0,60}within|within[^.]{0,40}days[^.]{0,40}refund", "Official format has no refund deadline, so the deposit can be delayed"),
          ("Electricity / water terms", r"electricity|water charge|utilit", "Unclear who pays bills and at what rate")]

def analyse_official(text, area_key, bhk):
    clauses, ctx, rows = split_official(text), {"area": area_key, "bhk": bhk}, []
    for i, (head, c) in enumerate(clauses, 1):
        ctype, risk, src, why = hybrid_predict_v2(c, ctx, final_risk_model)
        rows.append((i, head, c[:95] + ("..." if len(c) > 95 else ""), risk, src, why))
    rep = pd.DataFrame(rows, columns=["#", "Clause title", "Text", "Risk", "Decided by", "Why"])
    score, _ = risk_score(rep.Risk)
    missing = [(n, w) for n, pat, w in CHECKS if not re.search(pat, text, re.I)]
    return rep, score, missing

LAST = {}          # remembers the last analysed agreement (used by the question-search in Step 7)

def show_analysis(text, area_key, bhk):
    rep, score, missing = analyse_official(text, area_key, bhk)
    LAST.update(rep=rep, clauses=[c for h, c in split_official(text)])
    _, level = risk_score(rep.Risk)
    display(rep.style.apply(lambda col: [COL[v] for v in col], subset=["Risk"])
            .set_properties(subset=["Text", "Why"], **{"text-align": "left", "white-space": "normal"}))
    print(f"\nOVERALL RISK SCORE: {score}/100  ->  {level} RISK   |   High: {list(rep[rep.Risk=='High']['#'])}  Medium: {list(rep[rep.Risk=='Medium']['#'])}")
    poss = rep["Clause title"].eq("Possession") & rep.Risk.eq("High")
    if poss.any():
        s2, _ = risk_score(rep.Risk.where(~poss, "Low"))
        print(f"\nNOTE: the 'Possession' clause is FIXED wording of the official Maharashtra format (double daily damages, removal without going to court). It is flagged High in every agreement built on that format and adds 15 points; the score would be {s2} without it. Ask the owner to amend it.")
    if missing:
        print("\nNOT STATED in this agreement:")
        for n, w in missing: print(f"  - {n}: {w}")
    return rep, score

### 📸 6f. Three scripted scenarios (fair / borderline / risky) — also your fallback if the live form misbehaves

In [ ]:
SCEN = {
 "A. Fair agreement":      dict(area_name="Kothrud", bhk=2, months=11, mode="Fixed rent", rent=29000, rent2=0, m1=6, dep=58000, deptype="Refundable", maint="licensor", lock=6, lockwho="both", esc=5, fee=100, grace=7, refund=30),
 "B. Borderline":          dict(area_name="Wakad", bhk=2, months=11, mode="Fixed rent", rent=31000, rent2=0, m1=6, dep=87000, deptype="Refundable", maint="licensee", lock=8, lockwho="both", esc=8, fee=200, grace=5, refund=45),
 "C. Risky agreement":     dict(area_name="Viman Nagar", bhk=2, months=11, mode="Fixed rent", rent=55000, rent2=0, m1=6, dep=330000, deptype="Non-refundable", maint="licensee", lock=11, lockwho="licensee", esc=20, fee=1000, grace=0, refund=0),
}
KEY = {"Kothrud": "kothrud", "Wakad": "wakad", "Viman Nagar": "viman nagar"}
summary = []
for name, p in SCEN.items():
    print("\n" + "=" * 90 + f"\n{name}  ({p['area_name']}, {p['bhk']} BHK)\n" + "=" * 90)
    txt = build_agreement(**p); rep, score = show_analysis(txt, KEY[p["area_name"]], p["bhk"]); summary.append((name, score))
print("\nSUMMARY:", summary)

## 🎤 LIVE DEMO — fill the form
Fill only **rates and the terms the project checks** (no names, PAN, Aadhaar or signatures — the generated agreement uses *Licensor* and *Licensee* only). Then click **Analyse agreement**.
Leave an add-on at **0** to leave it *out* of the agreement — the system then lists it under *"NOT STATED"*.

**Two things to know before you present**
- **Rent rule:** the extra is counted from the **start (lowest rent)** of the Pune range for that area and BHK — up to Rs. 4,000 above the start is Low, up to Rs. 7,000 Medium, more is High. (Kothrud 2 BHK starts at Rs. 27,000, so Rs. 29,000 is Low, Rs. 33,000 Medium, Rs. 36,000 High.)
- **Possession is always High:** Clause 10 is *fixed wording of the government format* — double daily damages and removal without going to court. It is not an option on the form, so every agreement built on it carries this flag (the report prints the score *without* it). Treat it as a finding about the official form, not a bug.

In [ ]:
import ipywidgets as wd
from IPython.display import display as _d

sty, lay = {"description_width": "230px"}, wd.Layout(width="520px")
AREA_OPTS = [("Kothrud", "kothrud"), ("Hinjewadi Phase 1", "hinjewadi phase 1"), ("Hinjewadi Phase 2/3", "hinjewadi phase 2/3"),
             ("Sus Gaon", "sus gaon"), ("Wakad", "wakad"), ("Viman Nagar", "viman nagar"), ("Baner / Balewadi", "baner/balewadi"), ("Kharadi", "kharadi")]
F = dict(
 area  = wd.Dropdown(options=AREA_OPTS, description="Area", style=sty, layout=lay),
 bhk   = wd.Dropdown(options=[1, 2, 3], value=2, description="BHK", style=sty, layout=lay),
 months= wd.IntText(value=11, description="Period (months)", style=sty, layout=lay),
 rent  = wd.IntText(value=29000, description="Monthly rent Rs.", style=sty, layout=lay),
 staged= wd.Checkbox(value=False, description="Rent changes during the period (official staged-rent option)", indent=False, layout=wd.Layout(width="620px")),
 rent2 = wd.IntText(value=32000, description="Rent after the change Rs.", style=sty, layout=wd.Layout(width="520px", display="none")),
 m1    = wd.IntText(value=6, description="Months at the first rent", style=sty, layout=wd.Layout(width="520px", display="none")),
 dep   = wd.IntText(value=58000, description="Deposit Rs.", style=sty, layout=lay),
 deptype=wd.Dropdown(options=["Refundable", "Non-refundable"], description="Deposit type", style=sty, layout=lay),
 maint = wd.Dropdown(options=[("Licensor pays all outgoings", "licensor"), ("Licensee pays maintenance", "licensee")], description="Maintenance (Clause 4)", style=sty, layout=lay),
 lock  = wd.IntText(value=0, description="ADD-ON Lock-in months (0=none)", style=sty, layout=lay),
 lockwho=wd.Dropdown(options=[("Both parties", "both"), ("Licensee only", "licensee")], description="Lock-in applies to", style=sty, layout=lay),
 esc   = wd.FloatText(value=0, description="ADD-ON Yearly increase % (0=none)", style=sty, layout=lay),
 fee   = wd.IntText(value=0, description="ADD-ON Late fee Rs./day (0=none)", style=sty, layout=lay),
 grace = wd.IntText(value=7, description="Late-fee grace days", style=sty, layout=lay),
 refund= wd.IntText(value=0, description="ADD-ON Deposit refund within days (0=none)", style=sty, layout=lay))
btn, out = wd.Button(description="Analyse agreement", button_style="primary", layout=wd.Layout(width="250px", height="38px")), wd.Output()

def on_click(_):
    with out:
        out.clear_output(wait=True)
        v = {k: f.value for k, f in F.items()}
        area_name = dict((b, a) for a, b in AREA_OPTS)[v["area"]]
        txt = build_agreement(area_name, v["bhk"], v["months"], ("Staged rent" if v["staged"] else "Fixed rent"), v["rent"], v["rent2"], v["m1"], v["dep"], v["deptype"],
                              v["maint"], v["lock"], v["lockwho"], v["esc"], v["fee"], v["grace"], v["refund"])
        print("GENERATED AGREEMENT (official clause wording)\n" + "-" * 60); print(txt, "\n")
        show_analysis(txt, v["area"], v["bhk"])
def _toggle_staged(change):
    for k in ("rent2", "m1"): F[k].layout.display = "" if F["staged"].value else "none"
F["staged"].observe(_toggle_staged, names="value")
btn.on_click(on_click)
_d(wd.VBox(list(F.values()) + [btn, out]))

### Or paste a filled form
If you fill the official PDF text yourself, paste it below (delete the unused "OR" alternatives first), set the area and BHK, and run.

In [ ]:
pasted = """
LEAVE AND LICENSE AGREEMENT
1) Period: That the Licensor hereby grants to the Licensee a revocable leave and license for a period of 11 Months.
2) Rent & Deposit: That the Licensee shall pay to the Licensor Rs. 26,000 per month towards the compensation and Rs. 78,000 interest free refundable deposit, for the use of the said licensed premises.
3) Maintenance Charges: That the Licensee/s herein shall bear and pay all the maintenance charges in respect of the said Licensed Premises.
Schedule: ...
"""
show_analysis(pasted, "hinjewadi phase 2/3", 2);

## STEP 7 — Question Answering by Clause Search with Query Expansion  *(Practical 5)*

**Problem (vocabulary mismatch):** a tenant asks *"Who pays the light bill?"* but the agreement says *"electricity charges"*. A plain keyword/TF-IDF search finds **no shared words**, so it misses the clause. **Query expansion** adds related words (*light → electricity, power*) before searching.

**Five search variants are compared:**
| Variant | How the query is expanded |
|---|---|
| Baseline | no expansion (plain TF-IDF cosine similarity) |
| WordNet | synonyms from NLTK WordNet (general English) |
| Domain thesaurus | a small rental-law thesaurus (*landlord ↔ licensor, deposit ↔ security/advance, evict ↔ vacate/terminate …*) |
| WordNet + Domain | both |
| Domain + PRF | domain terms, then **pseudo-relevance feedback** (Rocchio): add the terms of the top-3 results and search again |

**Metrics** (24 tenant questions; a clause is *relevant* if its type matches the question's topic):
- **Precision@5** – of the top 5 results, how many are relevant
- **Recall@10** – of all relevant clauses, how many appear in the top 10
- **MRR** – 1 ÷ rank of the first relevant result (higher = the right clause comes first)

⚠️ Honest note: the questions and the thesaurus were both written by me, so the expansion gain is likely optimistic. Add **your own 10 questions** in the optional cell to test it fairly.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import wordnet as wn

# ---------- 7a. Domain thesaurus (general rental vocabulary) ----------
THES = {
 "landlord": ["licensor", "owner"], "owner": ["licensor", "landlord"], "tenant": ["licensee", "occupant"],
 "rent": ["compensation", "licence fee", "monthly"], "price": ["rent", "compensation"], "monthly": ["per month", "rent"],
 "deposit": ["security", "advance", "refundable"], "advance": ["deposit", "security"], "security": ["deposit", "refundable"],
 "return": ["refund"], "refund": ["return", "deposit"], "money": ["amount", "deposit"],
 "light": ["electricity", "power"], "power": ["electricity"], "electricity": ["power", "meter"], "water": ["metered water", "water charge"],
 "tax": ["property tax", "levy"], "bill": ["charge", "payment"],
 "evict": ["vacate", "terminate", "remove", "cancel"], "throw": ["evict", "remove", "vacate"], "leave": ["vacate", "terminate", "exit"],
 "move": ["vacate"], "lock": ["lock-in", "minimum stay", "commitment"], "minimum": ["lock-in", "commitment"], "stay": ["period", "lock-in"],
 "early": ["lock-in", "forfeit", "compensation"], "notice": ["written notice", "terminate"],
 "hike": ["increase", "revision", "escalation"], "raise": ["increase", "hike", "revise"], "increase": ["hike", "escalation"], "yearly": ["annual", "renewal"],
 "late": ["delay", "overdue", "penalty"], "delay": ["late", "fine", "penalty"], "fine": ["penalty", "late fee"], "penalty": ["fine", "late fee"],
 "repair": ["maintenance", "damage"], "damage": ["repair", "maintenance"], "painting": ["repaint", "maintenance"],
 "society": ["maintenance", "sinking fund", "corpus"], "maintenance": ["society", "repair"], "lock-in": ["minimum stay"],
}

def q_tokens(q): return preprocess(q).split()

def wordnet_expand(tokens, per_word=3):
    out = []
    for t in tokens:
        n = 0
        for syn in wn.synsets(t)[:2]:
            for l in syn.lemma_names()[:3]:
                l = l.lower()
                if "_" not in l and l != t and l not in out and n < per_word: out.append(l); n += 1
    return out

def domain_expand(tokens):
    out = []
    for t in tokens:
        for e in THES.get(t, []):
            if e not in out: out.append(e)
    return [w for e in out for w in preprocess(e).split()]

def rank_docs(question, vec, X, variant="baseline", alpha=0.5, beta=0.6):
    toks = q_tokens(question)
    q = vec.transform([" ".join(toks)]).toarray()
    exp = []
    if "wordnet" in variant: exp += wordnet_expand(toks)
    if "domain" in variant:  exp += domain_expand(toks)
    if exp: q = q + alpha * vec.transform([" ".join(exp)]).toarray()
    sims = cosine_similarity(q, X).ravel()
    if "prf" in variant:                                          # Rocchio pseudo-relevance feedback
        top = sims.argsort()[::-1][:3]
        q = q + beta * np.asarray(X[top].mean(axis=0))
        sims = cosine_similarity(q, X).ravel()
    return sims.argsort()[::-1], exp

# ---------- 7b. Evaluation set: 24 tenant questions -> relevant clause TYPE ----------
QUESTIONS = [
 ("How much do I have to pay every month for the flat?", "rent"), ("What is the monthly charge for staying in this apartment?", "rent"),
 ("How much money do I give as security and will I get it back?", "deposit"), ("When will the owner return my advance after I leave?", "deposit"),
 ("Can the owner keep my security money?", "deposit"),
 ("Can the owner raise the price next year?", "rent_increase"), ("By how much will the rent go up when I renew?", "rent_increase"),
 ("Is there a limit on yearly hikes?", "rent_increase"),
 ("What happens if I pay the rent late?", "penalty"), ("Will I be fined for delay in payment?", "penalty"),
 ("Are there any charges for breaking the rules?", "penalty"),
 ("Can I leave before the agreement ends?", "lockin"), ("What is the minimum period I must stay?", "lockin"),
 ("Do I have to pay if I move out early?", "lockin"),
 ("Can the landlord throw me out suddenly?", "eviction"), ("How much notice is needed to make me leave?", "eviction"),
 ("Can the owner lock the flat if I am late?", "eviction"),
 ("Who pays for repairs in the house?", "maintenance"), ("Do I have to pay society charges?", "maintenance"),
 ("Who bears the cost of painting and damage?", "maintenance"),
 ("Who pays the light bill?", "utilities"), ("How is water billed?", "utilities"),
 ("Do I pay property tax?", "utilities"), ("Can the owner cut off power supply?", "utilities"),
]
corpus = test.reset_index(drop=True)
vec_c = TfidfVectorizer(sublinear_tf=True).fit(corpus.clean_text)
X_c = vec_c.transform(corpus.clean_text).toarray()

def evaluate(variant, qs=QUESTIONS):
    P5, R10, RR = [], [], []
    for q, qtype in qs:
        order, _ = rank_docs(q, vec_c, X_c, variant)
        rel = (corpus.clause_type.values[order] == qtype); n_rel = (corpus.clause_type == qtype).sum()
        P5.append(rel[:5].mean()); R10.append(rel[:10].sum() / n_rel)
        RR.append(1 / (np.argmax(rel) + 1) if rel.any() else 0)
    return np.mean(P5), np.mean(R10), np.mean(RR)

VARIANTS = ["baseline", "wordnet", "domain", "wordnet+domain", "domain+prf"]
ev = pd.DataFrame([(v, *evaluate(v)) for v in VARIANTS], columns=["Variant", "Precision@5", "Recall@10", "MRR"]).round(3)
print(f"Corpus: {len(corpus)} clauses | Questions: {len(QUESTIONS)}")
display(ev)
ax = ev.set_index("Variant").plot(kind="bar", figsize=(9, 4), color=["#90a4ae", "#1e88e5", "#43a047"])
ax.set_ylim(0, 1); ax.set_title("Clause search: effect of query expansion"); plt.xticks(rotation=15); plt.tight_layout(); plt.show()

### 📸 7c. Per-topic breakdown and worked examples

In [ ]:
rows = []
for qtype in sorted(set(t for _, t in QUESTIONS)):
    qs = [(q, t) for q, t in QUESTIONS if t == qtype]
    rows.append((qtype, len(qs), evaluate("baseline", qs)[2], evaluate("domain", qs)[2], evaluate("domain+prf", qs)[2]))
display(pd.DataFrame(rows, columns=["Topic", "Questions", "MRR baseline", "MRR domain", "MRR domain+PRF"]).round(3))

def show_example(question, qtype, variant):
    order, exp = rank_docs(question, vec_c, X_c, variant)
    print(f"[{variant}]  expansion terms: {exp[:12]}")
    for r, i in enumerate(order[:3], 1):
        ok = "OK " if corpus.clause_type[i] == qtype else "-- "
        print(f"   {r}. {ok}({corpus.clause_type[i]:11s}) {corpus.clause_text[i][:85]}")
for question, qtype in [QUESTIONS[20], QUESTIONS[14], QUESTIONS[3]]:
    print("\nQUESTION:", question, f"   (relevant topic: {qtype})")
    show_example(question, qtype, "baseline"); show_example(question, qtype, "domain")

### 📸 7d. Which questions still fail? (error analysis)

In [ ]:
fail = []
for q, qtype in QUESTIONS:
    order, _ = rank_docs(q, vec_c, X_c, "domain")
    first = np.argmax(corpus.clause_type.values[order] == qtype) + 1
    if first > 1: fail.append((q, qtype, first, corpus.clause_type[order[0]]))
print(f"{len(fail)} of {len(QUESTIONS)} questions do not have a relevant clause at rank 1 (domain thesaurus):")
display(pd.DataFrame(fail, columns=["Question", "Topic", "First relevant rank", "Rank-1 topic returned"]))

### 7e. Ask questions about YOUR analysed agreement
The search now runs on the **clauses of the agreement you analysed** (Step 6) and shows each matching clause with its **risk colour** — so a tenant can ask *"Can the landlord throw me out suddenly?"* and immediately see Clause 10 flagged High. Run an analysis above first (the live-demo panel, or the scenario cell).

In [ ]:
def search_agreement(question, k=3, variant="domain"):
    if not LAST: print("Analyse an agreement first (Step 6 panel or scenario cell)."); return
    docs = [preprocess(c) for c in LAST["clauses"]]
    vec = TfidfVectorizer(sublinear_tf=True).fit(docs); X = vec.transform(docs).toarray()
    order, exp = rank_docs(question, vec, X, variant)
    res = LAST["rep"].iloc[order[:k]][["#", "Clause title", "Risk", "Why"]].copy()
    res.insert(1, "Matching text", [LAST["clauses"][i][:110] + "..." for i in order[:k]])
    print(f"QUESTION: {question}\nexpanded with: {exp[:10]}")
    display(res.style.apply(lambda col: [COL[v] for v in col], subset=["Risk"])
            .set_properties(subset=["Matching text", "Why"], **{"text-align": "left", "white-space": "normal"}))

# demo on the risky scenario C
p = SCEN["C. Risky agreement"]; _ = show_analysis(build_agreement(**p), KEY[p["area_name"]], p["bhk"])
for qn in ["Can the landlord throw me out suddenly?", "Will I get my security money back?", "Who pays for the society charges?"]:
    search_agreement(qn)

### 🎤 Live: type any question

In [ ]:
q_box = wd.Text(placeholder="e.g. Who pays the light bill?", description="Question", layout=wd.Layout(width="600px"))
q_btn, q_out = wd.Button(description="Search", button_style="success"), wd.Output()
def on_ask(_):
    with q_out: q_out.clear_output(wait=True); search_agreement(q_box.value)
q_btn.on_click(on_ask); _d(wd.VBox([q_box, q_btn, q_out]))

### Optional: test with YOUR OWN questions (fair evaluation)
Write 10 questions the way a real tenant would ask and give the topic (`rent, deposit, rent_increase, penalty, lockin, eviction, maintenance, utilities`). Better still, let a classmate write them without seeing my thesaurus.

In [ ]:
my_questions = [
    # ("Who will fix a leaking tap?", "maintenance"),
    # ("Is the advance money returned?", "deposit"),
]
if my_questions:
    print(pd.DataFrame([(v, *evaluate(v, my_questions)) for v in VARIANTS], columns=["Variant", "Precision@5", "Recall@10", "MRR"]).round(3))
else:
    print("Add questions to my_questions above and re-run this cell.")

## ✅ Progress so far
| Step | Status |
|---|---|
| 1. Problem, Pune rules, dataset | Done |
| 2. NLTK preprocessing | Done |
| 3. Clause splitting + spaCy NER | Done |
| 4. TF-IDF + 3-model comparison, unseen-test evaluation | Done |
| 5. Hybrid rules+ML risk engine with explanations | Done |
| 6. Official-form mode: engine v2 + live fill-the-form demo | Done |
| 7. Query-expansion clause search + evaluation (Practical 5) | Done |
| 8. Extras: missing-clause detector, translation, PDF export | *next* |